### Google authentication and Imports

In [1]:
# --- STEP 1: install packages, then restart the runtime ---
# Run this notebook on Google Colab (Python 3.12) so precompiled wheels are used.
# After this cell finishes, the runtime restarts automatically; then run the
# next cell (Step 2) to do the patch, imports, and BigQuery auth.
%pip install "spacy>=3.8,<4.0"
%pip install "scispacy==0.6.2"
%pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz
%pip install medspacy
%pip install gensim

# Confirm the model actually installed before restarting.
import importlib.util
assert importlib.util.find_spec("en_core_sci_sm") is not None, \
    "en_core_sci_sm did not install correctly -- check the pip output above for errors."
print("en_core_sci_sm is installed. Restarting runtime...")

import os
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 89.3 MB/s eta 0:00:00:00:01:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 85.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 23.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
to

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 9.5 MB/s eta 0:00:00:00:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 88.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.0/865.0 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 90.3 MB/s eta 0:00:00:00:010:01
  Created wheel for en_core_sci_sm: filename=en_core_sci_sm-0.5.4-py3-none-any.whl size=14778488 sha256=87bbf6c59d61fc392be789e125a4393a5c234899de9dc04173233a3a296c0e91
  Stored in directory: /root/.cache/pip/wheels/49/7f/0f/ec0fc3a935bfe55e6ef2ca04b7a31e33cbd533a6d7cbd9e11e
Successfully built en_core_sci_sm
  Attempting uninstall: blis
    Found existing installation: blis 1.3.3
    Uninstalling blis-1.3.3:
      Successfully uninstalled blis-1.3

: 

: 

: 

In [1]:
# --- STEP 2: run AFTER the runtime restart from Step 1 ---
# Patch the en_core_sci_sm 0.5.4 config so spaCy 3.8 accepts it, then import
# everything and authenticate to BigQuery.
import os, re, en_core_sci_sm

model_dir = os.path.dirname(en_core_sci_sm.__file__)
pattern = re.compile(r'include_static_vectors\s*=\s*"?(False|True)"?', re.IGNORECASE)
patched = []
for root, _, files in os.walk(model_dir):
    for fname in files:
        if fname == "config.cfg":
            p = os.path.join(root, fname)
            with open(p, "r", encoding="utf-8") as fh:
                text = fh.read()
            new_text, n = pattern.subn(
                lambda m: f"include_static_vectors = {m.group(1).lower()}", text
            )
            if n and new_text != text:
                with open(p, "w", encoding="utf-8") as fh:
                    fh.write(new_text)
                patched.append(p)
print("Patched config files:", patched)

from google.colab import auth
from google.cloud import bigquery
import scispacy
import medspacy
import spacy
import gensim
import gensim.downloader as api
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from gensim.models import Word2Vec

auth.authenticate_user()
project_id = 'ai-on-healthcare'
client = bigquery.Client(project=project_id)

print("spacy:", spacy.__version__, "| scispacy:", scispacy.__version__, "| medspacy:", medspacy.__version__)

/usr/local/lib/python3.12/dist-packages/spacy/util.py:971: UserWarning: [W095] Model 'en_core_sci_sm' (0.5.4) was trained with spaCy v3.7.4 and may not be 100% compatible with the current version (3.8.14). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


Patched config files: ['/usr/local/lib/python3.12/dist-packages/en_core_sci_sm/en_core_sci_sm-0.5.4/config.cfg']


MessageError: Failed to issue request POST https://colab.research.google.com/tun/m/credentials-propagation/m-s-kkb-euw4b1-35v6jhn2d1azi?authtype=auth_user_ephemeral&version=2&dryrun=false&propagate=true&record=false&authuser=0: Bad Request
Response body: 
<!DOCTYPE html>
<html lang=en>
  <meta charset=utf-8>
  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">
  <title>Error 400 (Bad Request)!!1</title>
  <style>
    *{margin:0;padding:0}html,code{font:15px/22px arial,sans-serif}html{background:#fff;color:#222;padding:15px}body{margin:7% auto 0;max-width:390px;min-height:180px;padding:30px 0 15px}* > body{background:url(//www.google.com/images/errors/robot.png) 100% 5px no-repeat;padding-right:205px}p{margin:11px 0 22px;overflow:hidden}ins{color:#777;text-decoration:none}a img{border:0}@media screen and (max-width:772px){body{background:none;margin-top:0;max-width:none;padding-right:0}}#logo{background:url(//www.google.com/images/logos/errorpage/error_l

### Diagnoses Query

In [ ]:
ICD_FILTER = ['430']

diagnose_query = """
    SELECT *
    FROM `physionet-data.mimiciii_clinical.diagnoses_icd`
    LIMIT 100000
"""

# Run the query and convert the results to a Pandas DataFrame
df_diagnoses = client.query(diagnose_query).to_dataframe()
df_diagnoses = df_diagnoses[df_diagnoses['ICD9_CODE'].isin(ICD_FILTER)]
subj_filtered = df_diagnoses['SUBJECT_ID'].unique()
print(subj_filtered)

### Get notes data

In [ ]:
notes_query = f"""
    SELECT SUBJECT_ID, HADM_ID, CATEGORY, TEXT
    FROM `physionet-data.mimiciii_notes.noteevents`
    WHERE SUBJECT_ID IN UNNEST(@subject_ids)
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ArrayQueryParameter(
            "subject_ids", "INT64", [int(s) for s in subj_filtered]
        )
    ]
)

df_notes = client.query(notes_query, job_config=job_config).to_dataframe()
df_notes_filtered = df_notes.dropna(subset=["TEXT"]).reset_index(drop=True)
print(f"Notes retrieved: {len(df_notes_filtered)} for {df_notes_filtered['SUBJECT_ID'].nunique()} subjects")
df_notes_filtered.head()

### Get pretrained model

In [ ]:
info = api.info()  # show info about available models/datasets
pretrained_model= api.load("glove-wiki-gigaword-50")  # download the model and return as object ready for use

### Function to graph T-SNE

In [ ]:
import numpy as np

def tsne_plot(model,words, preTrained=False):
    "Creates and TSNE model and plots it"
    labels = []
    tokens = []

    for word in words:
      if preTrained:
          tokens.append(model[word])
      else:
          tokens.append(model.wv[word])
      labels.append(word)

    tokens = np.array(tokens)
    tsne_model = TSNE(perplexity=30, early_exaggeration=12, n_components=2, init='pca', n_iter=1000, random_state=23)
    new_values = tsne_model.fit_transform(tokens)

    x = []
    y = []
    for value in new_values:
        x.append(value[0])
        y.append(value[1])

    plt.figure(figsize=(16, 16))
    for i in range(len(x)):
        plt.scatter(x[i],y[i])
        plt.annotate(labels[i],
                     xy=(x[i], y[i]),
                     xytext=(5, 2),
                     textcoords='offset points',
                     ha='right',
                     va='bottom')
    plt.show()

### Extract Entities with Spacy

In [ ]:
nlp = spacy.load("en_core_web_sm")
corpus = []
for row in range(0, len(df_notes_filtered)):
    ents = nlp(df_notes_filtered.iloc[row]['TEXT']).ents
    str_tokens = [t.text for t in ents]
    if str_tokens:  # skip notes with no entities
        corpus.append(str_tokens)

print(f"Documents in corpus: {len(corpus)}")
assert corpus, "Corpus is empty -- check df_notes_filtered has rows with TEXT"

### Graph embbedings the corpus using t-SNE and Spacy

In [ ]:
model1 = Word2Vec(corpus, min_count=1)
vocabs = model1.wv.key_to_index.keys()
new_v = np.array(list(vocabs))
tsne_plot(model1,new_v)

### Extract Entities with SciSpacy

In [ ]:
nlp_scispacy = spacy.load("en_core_sci_sm")

corpus_sci = []
for row in range(0, len(df_notes_filtered)):
    ents = nlp_scispacy(df_notes_filtered.iloc[row]['TEXT']).ents
    str_tokens = [t.text for t in ents]
    if str_tokens:
        corpus_sci.append(str_tokens)

print(f"Documents in corpus_sci: {len(corpus_sci)}")
assert corpus_sci, "corpus_sci is empty"

### Graph embbedings the corpus using t-SNE and SciSpacy

In [ ]:
model1 = Word2Vec(corpus_sci, min_count=1)
vocabs = model1.wv.key_to_index.keys()
new_v = np.array(list(vocabs))
tsne_plot(model1,new_v)

### Extract Entities using MedSciSpacy

In [ ]:
# medspaCy adds a PyRuSH sentencizer that clashes with the scispacy parser
# (both want to set token.sent_start, producing "ValueError [E043]").
# Since we only need entities here, disable the medspaCy sentencizer and let
# the parser do sentence splitting.
nlp_medspacy = medspacy.load("en_core_sci_sm", disable=["medspacy_pyrush"])
print("medspaCy pipeline:", nlp_medspacy.pipe_names)

corpus_med = []
for row in range(0, len(df_notes_filtered)):
    ents = nlp_medspacy(df_notes_filtered.iloc[row]['TEXT']).ents
    str_tokens = [t.text for t in ents]
    if str_tokens:
        corpus_med.append(str_tokens)

print(f"Documents in corpus_med: {len(corpus_med)}")
print("Sample entities from first note:", corpus_med[0][:20] if corpus_med else "empty")
assert corpus_med, "corpus_med is empty"